## 1

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Use GPU 1 which has 40GB free
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Reduce fragmentation

# Then your existing imports and code...

import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, RobertaModel,
                          RobertaConfig, RobertaForCausalLM)
import timm
import Levenshtein
from tqdm import tqdm

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_distil.pth"

# =====================================================================
# 2. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 transform=None, max_target_length=25):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                status        = parts[1]
                if status != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts           = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent      = f"{id_parts[0]}-{id_parts[1]}"
                filename           = f"{word_id}.png"
                img_path = os.path.join(
                    self.data_dir, "words",
                    folder_grandparent, folder_parent, filename
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images appear corrupted.")
        if self.transform:
            image = self.transform(image)
        labels = self.processor(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        labels = torch.where(
            labels == self.processor.pad_token_id,
            torch.tensor(-100), labels
        )
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((64, 256)),
        T.RandomApply([T.ColorJitter(
            brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(
        data_dir, words_txt_path,
        processor=processor, transform=None
    )
    total      = len(full_dataset)
    train_size = int(0.9 * total)

    generator     = torch.Generator().manual_seed(42)
    indices       = torch.randperm(total, generator=generator).tolist()
    train_indices = indices[:train_size]
    val_indices   = indices[train_size:]

    train_samples = [full_dataset.samples[i] for i in train_indices]
    val_samples   = [full_dataset.samples[i] for i in val_indices]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor,
                     transform, max_target_length=25):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (256, 64), color=255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_dataset = SplitDataset(
        train_samples, processor, train_transform
    )
    val_dataset   = SplitDataset(
        val_samples, processor, val_transform
    )
    train_loader  = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=4, drop_last=True
    )
    val_loader    = DataLoader(
        val_dataset, batch_size=batch_size,
        shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE
# =====================================================================

# ── TPS-STN ──────────────────────────────────────────────────────────
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B    = x.size(0)
        pts  = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:, :, 0].mean(dim=1)
        cy = control_points[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False,
                             padding_mode='border')


# ── Swin Encoder ─────────────────────────────────────────────────────
class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)  # (B, ~64, 512)
        return self.proj(feat)             # (B, ~64, 768)


# ── DistilRoBERTa Decoder ─────────────────────────────────────────────
class DistilRoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        print("Loading DistilRoBERTa pretrained weights...")

        distilroberta = RobertaModel.from_pretrained(
            "distilroberta-base"
        )
        config        = RobertaConfig.from_pretrained(
            "distilroberta-base"
        )

        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size

        self.decoder = RobertaForCausalLM(config)

        self.decoder.roberta.embeddings.load_state_dict(
            distilroberta.embeddings.state_dict()
        )

        for i, layer in enumerate(
                self.decoder.roberta.encoder.layer):
            layer.load_state_dict(
                distilroberta.encoder.layer[i].state_dict(),
                strict=False  
            )

        del distilroberta
        print("DistilRoBERTa: all 6 layers loaded successfully.")

    def forward(self, input_ids,
                encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


# ── Full Pipeline ─────────────────────────────────────────────────────
class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768,
                 num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.decoder = DistilRoBERTaDecoder(
            vocab_size=vocab_size, d_model=d_model
        )

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)          # (B, 3, H, W)
        swin_feat = self.swin_encoder(rectified)  # (B, seq, 768)
        memory, _ = self.bilstm(swin_feat)         # (B, seq, 768)
        return memory

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),  # pad_token_id = 1
            dec_input
        )

        labels = target_ids[:, 1:].clone()

        outputs = self.decoder(
            input_ids             = dec_input,
            encoder_hidden_states = memory,
            labels                = None   
        )

        logits = outputs.logits  # (B, T-1, vocab)

        if criterion is not None:
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        else:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100
            )
        return loss

    @torch.no_grad()
    def generate(self, images, max_length=25,
                 bos_token_id=0, eos_token_id=2):
        device    = images.device
        B         = images.size(0)
        memory    = self._extract_visual_memory(images)
        generated = torch.full(
            (B, 1), bos_token_id,
            dtype=torch.long, device=device
        )
        for _ in range(max_length - 1):
            outputs     = self.decoder(
                input_ids             = generated,
                encoder_hidden_states = memory,
                labels                = None
            )
            next_tokens = outputs.logits[:, -1, :].argmax(
                dim=-1, keepdim=True
            )
            generated   = torch.cat([generated, next_tokens], dim=1)
            if (generated == eos_token_id).any(dim=1).all():
                break
        return generated


# =====================================================================
# 4. AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = set()
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        self.lexicon.add(word)
        print(f"Lexicon built: {len(self.lexicon)} words.")

    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
        best_candidate = text_output
        min_distance   = float('inf')
        target_len     = len(cleaned)
        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist < min_distance and dist == 1:
                min_distance   = dist
                best_candidate = word
        if min_distance == 1:
            return (best_candidate.upper()
                    if text_output.isupper()
                    else best_candidate)
        return text_output


# =====================================================================
# 5. METRICS & EARLY STOPPING
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        pred_c   = pred.strip()
        target_c = target.strip()
        dist     = Levenshtein.distance(pred_c, target_c)
        if len(target_c) > 0:
            total_cer += dist / len(target_c)
        elif len(pred_c) > 0:
            total_cer += 1.0
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    """
    Early stops the training if validation metric (WER) doesn't improve after a given patience.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_metric = float('inf')
        self.early_stop = False

    def __call__(self, current_metric):
        # We look for a decrease in WER
        if current_metric < self.best_metric - self.min_delta:
            self.best_metric = current_metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 6. TRAINING
# =====================================================================
def run_training_pipeline(epochs=20, batch_size=32, lr=5e-5, early_stopping_patience=4):
    print("Loading RoBERTa tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

    print(f"BOS={tokenizer.bos_token_id} | "
          f"EOS={tokenizer.eos_token_id} | "
          f"PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE,
        tokenizer, batch_size=batch_size
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model  = CompleteHTRPipeline(
        vocab_size=tokenizer.vocab_size
    ).to(device)

    total_params   = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    print(f"Total params    : {total_params/1e6:.2f}M")
    print(f"Trainable params: {trainable_params/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100,
        label_smoothing=0.1
    )

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.05
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5,
        patience=2, min_lr=1e-6, verbose=True
    )

    # Initialize early stopping tracking validation WER
    early_stopper = EarlyStopping(patience=early_stopping_patience, min_delta=0.0)

    agent_verifier    = AgenticVerificationModule(
        words_txt_file=WORDS_TXT_FILE
    )
    best_val_wer      = float('inf')
    swin_unfrozen     = False

    print(f"\nTraining on: {device}")

    for epoch in range(1, epochs + 1):

        # Unfreeze Swin at epoch 4
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(),
                 'lr': 1e-6},
                {'params': model.swin_encoder.proj.parameters(),
                 'lr': lr},
                {'params': model.bilstm.parameters(),
                 'lr': lr},
                {'params': model.decoder.parameters(),
                 'lr': lr},
                {'params': model.tps_stn.parameters(),
                 'lr': lr},
            ], weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5,
                patience=2, min_lr=1e-6, verbose=True
            )
            swin_unfrozen = True
            print("Swin encoder unfrozen.")

        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels,
                         criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(
                    val_loader,
                    desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens = model.generate(
                    images,
                    max_length   = 25,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id
                )

                # Debug first batch each epoch
                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(
                            tokens[i],
                            skip_special_tokens=True
                        )
                        print(f"Target: '{text_labels[i]}' "
                              f"| Pred: '{raw.strip()}'")
                    print("----------------------------\n")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(
                        tokens[i], skip_special_tokens=True
                    )
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        scheduler.step(current_wer)
        print(f"Epoch {epoch} | "
              f"CER: {metrics['CER']:.2f}% | "
              f"WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer':             best_val_wer,
                'cer':                  metrics['CER']
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | WER: {best_val_wer:.2f}%")
        
        # Check early stopping condition
        if early_stopper(current_wer):
            print(f"\nEarly stopping triggered! Training stopped at epoch {epoch}.")
            break
            
        print("=" * 70)


if __name__ == "__main__":
    run_training_pipeline(epochs=50, batch_size=32, lr=5e-5, early_stopping_patience=4)

Loading RoBERTa tokenizer...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BOS=0 | EOS=2 | PAD=1
Registered 38305 samples.


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loading DistilRoBERTa pretrained weights...
DistilRoBERTa: all 6 layers loaded successfully.


...

Epoch 50 | Train Loss: 1.4656


Epoch 50/50 [Val]:   2%|█▎                                                | 3/120 [00:00<00:12,  9.41it/s]


--- Debug Epoch 50 ---
Target: 'any' | Pred: 'any'
Target: 'any' | Pred: 'any'
Target: 'unopposed' | Pred: 'proposed'
----------------------------



Epoch 50/50 [Val]: 100%|████████████████████████████████████████████████| 120/120 [00:09<00:00, 12.53it/s]

Epoch 50 | CER: 13.90% | WER: 22.71%
EarlyStopping counter: 2 out of 4


In [3]:
#test
import os
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer
import Levenshtein
from tqdm import tqdm

# Import your model and module classes here if they are in a different file
# from train_script import CompleteHTRPipeline, AgenticVerificationModule, calculate_metrics

# =====================================================================
# 1. CONFIGURATION & PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_distil.pth"
BATCH_SIZE      = 32

os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

# =====================================================================
# 2. TEST DATASET (Matches your training split strategy)
# =====================================================================
class IAMTestDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.processor = processor
        self.samples = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9 or parts[1] != "ok":
                    continue
                
                word_id = parts[0]
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                id_parts = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent      = f"{id_parts[0]}-{id_parts[1]}"
                filename           = f"{word_id}.png"
                img_path = os.path.join(
                    self.data_dir, "words",
                    folder_grandparent, folder_parent, filename
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))

def get_test_loader(data_dir, words_txt_path, processor, batch_size=32):
    # Standard clean validation/testing transform (No training augmentations)
    test_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    # Re-use your original dataset class to parse the files flawlessly
    full_dataset = IAMWordsDataset(
        data_dir, words_txt_path,
        processor=processor, transform=None
    )
    total = len(full_dataset)
    train_size = int(0.9 * total)

    # Recreate the exact same deterministic split from training
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(total, generator=generator).tolist()
    val_indices = indices[train_size:]  
    val_samples = [full_dataset.samples[i] for i in val_indices]

    # Clean subset dataset for evaluation
    class SplitTestDataset(Dataset):
        def __init__(self, samples, transform):
            self.samples = samples
            self.transform = transform

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (256, 64), color=255)
            if self.transform:
                image = self.transform(image)
            return image, text

    test_dataset = SplitTestDataset(val_samples, test_transform)
    return DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# =====================================================================
# 3. EVALUATION FUNCTION
# =====================================================================
def run_evaluation():
    print("Initializing components...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Initialize Model Architecture
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    
    # Load Model Weights
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"No checkpoint found at {CHECKPOINT_PATH}")
        
    print(f"Loading weights from: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded model from epoch {checkpoint.get('epoch', 'N/A')} (Best Train/Val WER: {checkpoint.get('best_wer', 'N/A')}%)")
    
    # Setup Data Loader
    test_loader = get_test_loader(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=BATCH_SIZE)
    
    # Setup Agentic Verifier
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    # Arrays to hold predictions
    raw_predictions = []
    verified_predictions = []
    ground_truths = []
    
    model.eval()
    print("\nRunning inference on test split...")
    
    with torch.no_grad():
        for images, text_labels in tqdm(test_loader, desc="[Testing]"):
            images = images.to(device)
            
            # Predict
            tokens = model.generate(
                images,
                max_length=25,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
            # Decode and Post-process
            for i in range(images.size(0)):
                raw_pred = tokenizer.decode(tokens[i], skip_special_tokens=True).strip()
                verified_pred = agent_verifier.verify_and_correct(raw_pred).strip()
                
                raw_predictions.append(raw_pred)
                verified_predictions.append(verified_pred)
                ground_truths.append(text_labels[i])

    # Calculate Metrics
    raw_metrics = calculate_metrics(raw_predictions, ground_truths)
    verified_metrics = calculate_metrics(verified_predictions, ground_truths)
    
    # =====================================================================
    # 4. REPORT DISPLAY
    # =====================================================================
    print("\n" + "="*50)
    print("               FINAL TEST METRICS               ")
    print("="*50)
    print(f"Total Evaluated Samples: {len(ground_truths)}")
    print("-"*50)
    print("RAW MODEL OUTPUT (Without Agentic Correction):")
    print(f"  ↳ Character Error Rate (CER): {raw_metrics['CER']}%")
    print(f"  ↳ Word Error Rate (WER)     : {raw_metrics['WER']}%")
    print("-"*50)
    print("VERIFIED OUTPUT (With Agentic Correction Module):")
    print(f"  ↳ Character Error Rate (CER): {verified_metrics['CER']}%")
    print(f"  ↳ Word Error Rate (WER)     : {verified_metrics['WER']}%")
    print("="*50)

    # Show a few qualitative examples
    print("\n--- Sample Predictions Comparison ---")
    for i in range(min(5, len(ground_truths))):
        print(f"Ground Truth : '{ground_truths[i]}'")
        print(f"Raw Model    : '{raw_predictions[i]}'")
        print(f"Agent Fixed  : '{verified_predictions[i]}'")
        print("-" * 35)

if __name__ == "__main__":
    run_evaluation()

Initializing components...
Loading DistilRoBERTa pretrained weights...
DistilRoBERTa: all 6 layers loaded successfully.
Loading weights from: /home/mca24-26/handwritten text detection/iam_htr_distil.pth


/tmp/ipykernel_3418153/2969982914.py:126: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)


Loaded model from epoch 48 (Best Train/Val WER: 22.4223%)
Registered 38305 samples.
Lexicon built: 6231 words.

Running inference on test split...


[Testing]: 100%|████████████████████████████████████████████████████████| 120/120 [00:09<00:00, 12.47it/s]



               FINAL TEST METRICS               
Total Evaluated Samples: 3831
--------------------------------------------------
RAW MODEL OUTPUT (Without Agentic Correction):
  ↳ Character Error Rate (CER): 13.7579%
  ↳ Word Error Rate (WER)     : 22.5007%
--------------------------------------------------
VERIFIED OUTPUT (With Agentic Correction Module):
  ↳ Character Error Rate (CER): 13.8239%
  ↳ Word Error Rate (WER)     : 22.4223%

--- Sample Predictions Comparison ---
Ground Truth : 'any'
Raw Model    : 'any'
Agent Fixed  : 'any'
-----------------------------------
Ground Truth : 'any'
Raw Model    : 'any'
Agent Fixed  : 'any'
-----------------------------------
Ground Truth : 'unopposed'
Raw Model    : 'imposed'
Agent Fixed  : 'imposed'
-----------------------------------
Ground Truth : 'conference'
Raw Model    : 'conference'
Agent Fixed  : 'conference'
-----------------------------------
Ground Truth : 'Sir'
Raw Model    : 'Sir'
Agent Fixed  : 'Sir'
------------------------

## 2 (fix: beam search, lr and scheduler, lexicon)

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Use GPU 1 which has 40GB free
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Reduce fragmentation

# Then your existing imports and code...

import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, RobertaModel,
                          RobertaConfig, RobertaForCausalLM)
import timm
import Levenshtein
from tqdm import tqdm

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_distil_2.pth"

# =====================================================================
# 2. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 transform=None, max_target_length=25):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                status        = parts[1]
                if status != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts           = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent      = f"{id_parts[0]}-{id_parts[1]}"
                filename           = f"{word_id}.png"
                img_path = os.path.join(
                    self.data_dir, "words",
                    folder_grandparent, folder_parent, filename
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images appear corrupted.")
        if self.transform:
            image = self.transform(image)
        labels = self.processor(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        labels = torch.where(
            labels == self.processor.pad_token_id,
            torch.tensor(-100), labels
        )
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((64, 256)),
        T.RandomApply([T.ColorJitter(
            brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(
        data_dir, words_txt_path,
        processor=processor, transform=None
    )
    total      = len(full_dataset)
    train_size = int(0.9 * total)

    generator     = torch.Generator().manual_seed(42)
    indices       = torch.randperm(total, generator=generator).tolist()
    train_indices = indices[:train_size]
    val_indices   = indices[train_size:]

    train_samples = [full_dataset.samples[i] for i in train_indices]
    val_samples   = [full_dataset.samples[i] for i in val_indices]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor,
                     transform, max_target_length=25):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (256, 64), color=255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_dataset = SplitDataset(
        train_samples, processor, train_transform
    )
    val_dataset   = SplitDataset(
        val_samples, processor, val_transform
    )
    train_loader  = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=4, drop_last=True
    )
    val_loader    = DataLoader(
        val_dataset, batch_size=batch_size,
        shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE
# =====================================================================

# ── TPS-STN ──────────────────────────────────────────────────────────
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B    = x.size(0)
        pts  = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:, :, 0].mean(dim=1)
        cy = control_points[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False,
                             padding_mode='border')


# ── Swin Encoder ─────────────────────────────────────────────────────
class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)  # (B, ~64, 512)
        return self.proj(feat)             # (B, ~64, 768)


# ── DistilRoBERTa Decoder ─────────────────────────────────────────────
class DistilRoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        print("Loading DistilRoBERTa pretrained weights...")

        distilroberta = RobertaModel.from_pretrained(
            "distilroberta-base"
        )
        config        = RobertaConfig.from_pretrained(
            "distilroberta-base"
        )

        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size

        self.decoder = RobertaForCausalLM(config)

        self.decoder.roberta.embeddings.load_state_dict(
            distilroberta.embeddings.state_dict()
        )

        for i, layer in enumerate(
                self.decoder.roberta.encoder.layer):
            layer.load_state_dict(
                distilroberta.encoder.layer[i].state_dict(),
                strict=False  
            )

        del distilroberta
        print("DistilRoBERTa: all 6 layers loaded successfully.")

    def forward(self, input_ids,
                encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


# ── Full Pipeline ─────────────────────────────────────────────────────
class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768,
                 num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.decoder = DistilRoBERTaDecoder(
            vocab_size=vocab_size, d_model=d_model
        )

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)          # (B, 3, H, W)
        swin_feat = self.swin_encoder(rectified)  # (B, seq, 768)
        memory, _ = self.bilstm(swin_feat)         # (B, seq, 768)
        return memory

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),  # pad_token_id = 1
            dec_input
        )

        labels = target_ids[:, 1:].clone()

        outputs = self.decoder(
            input_ids             = dec_input,
            encoder_hidden_states = memory,
            labels                = None   
        )

        logits = outputs.logits  # (B, T-1, vocab)

        if criterion is not None:
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        else:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100
            )
        return loss

    @torch.no_grad()
    def generate(self, images, max_length=25,
                 bos_token_id=0, eos_token_id=2,
                 beam_size=5):
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)
    
        # Process one sample at a time for beam search
        all_results = []
    
        for b in range(B):
            mem   = memory[b:b+1]  # (1, seq, 768)
            # Each beam: (score, token_list)
            beams = [(0.0, [bos_token_id])]
            completed = []
    
            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor(
                        [tokens], dtype=torch.long, device=device
                    )
                    out     = self.decoder(
                        input_ids             = inp,
                        encoder_hidden_states = mem,
                        labels                = None
                    )
                    log_probs = F.log_softmax(
                        out.logits[0, -1], dim=-1
                    )
                    # Take top beam_size tokens
                    top_log_probs, top_ids = log_probs.topk(beam_size)
                    for log_p, tok_id in zip(
                            top_log_probs.tolist(),
                            top_ids.tolist()):
                        candidates.append(
                            (score + log_p, tokens + [tok_id])
                        )
    
                if not candidates:
                    break
    
                # Keep top beam_size candidates
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = candidates[:beam_size]
    
                # Move completed beams out
                still_running = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        still_running.append((score, tokens))
                beams = still_running
    
                if not beams:
                    break
    
            # Pick best completed or best running beam
            all_finished = completed if completed else beams
            best_score, best_tokens = max(
                all_finished, key=lambda x: x[0]
            )
            # Pad to max_length for consistent batching
            best_tokens = best_tokens[1:]  # remove BOS
            result = torch.tensor(
                best_tokens, dtype=torch.long, device=device
            )
            all_results.append(result)
    
        # Pad results to same length for batch return
        max_len = max(r.size(0) for r in all_results)
        padded  = torch.full(
            (B, max_len), eos_token_id,
            dtype=torch.long, device=device
        )
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
    
        return padded


# =====================================================================
# 4. AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = set()
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip()
                    if word:
                        # Store both original and lowercase
                        self.lexicon.add(word.lower())
        print(f"Lexicon built: {len(self.lexicon)} words.")
    
    def verify_and_correct(self, text_output):
        cleaned    = text_output.strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
    
        best_candidate = text_output
        min_distance   = float('inf')
        target_len     = len(cleaned)
    
        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist < min_distance and dist == 1:
                min_distance   = dist
                best_candidate = word
    
        if min_distance == 1:
            # Preserve original casing style
            if text_output.isupper():
                return best_candidate.upper()
            elif text_output[0].isupper():
                return best_candidate.capitalize()
            return best_candidate
        return text_output


# =====================================================================
# 5. METRICS & EARLY STOPPING
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        pred_c   = pred.strip()
        target_c = target.strip()
        dist     = Levenshtein.distance(pred_c, target_c)
        if len(target_c) > 0:
            total_cer += dist / len(target_c)
        elif len(pred_c) > 0:
            total_cer += 1.0
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    """
    Early stops the training if validation metric (WER) doesn't improve after a given patience.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_metric = float('inf')
        self.early_stop = False

    def __call__(self, current_metric):
        # We look for a decrease in WER
        if current_metric < self.best_metric - self.min_delta:
            self.best_metric = current_metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 6. TRAINING
# =====================================================================
def run_training_pipeline(epochs=80, batch_size=32,
                          lr=5e-5, early_stopping_patience=8,
                          resume_from=None):
    print("Loading RoBERTa tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

    print(f"BOS={tokenizer.bos_token_id} | "
          f"EOS={tokenizer.eos_token_id} | "
          f"PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE,
        tokenizer, batch_size=batch_size
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model  = CompleteHTRPipeline(
        vocab_size=tokenizer.vocab_size
    ).to(device)

    total_params   = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    print(f"Total params    : {total_params/1e6:.2f}M")
    print(f"Trainable params: {trainable_params/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100,
        label_smoothing=0.1
    )

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.05
    )
    # Replace scheduler with more patient version
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        factor   = 0.7,    # gentler reduction (was 0.5)
        patience = 4,      # wait longer before reducing (was 2)
        min_lr   = 5e-7,
        verbose  = True
    )
    
    # Increase early stopping patience
    early_stopper = EarlyStopping(
        patience  = 8,     # was 4 — too aggressive
        min_delta = 0.1    # only trigger if improvement < 0.1%
    )

    agent_verifier    = AgenticVerificationModule(
        words_txt_file=WORDS_TXT_FILE
    )
    best_val_wer      = float('inf')
    swin_unfrozen     = False

    print(f"\nTraining on: {device}")

    start_epoch    = 1
    best_val_wer   = float('inf')

    # Resume from checkpoint if provided
    if resume_from and os.path.exists(resume_from):
        print(f"Resuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1

        # Since we trained past epoch 4, Swin is already unfrozen
        if start_epoch > 4:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            swin_unfrozen = True
            # Rebuild optimizer with all params at lower LR
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(),
                 'lr': 5e-7},   # very low for already-trained Swin
                {'params': model.swin_encoder.proj.parameters(),
                 'lr': 1e-5},
                {'params': model.bilstm.parameters(),
                 'lr': 1e-5},
                {'params': model.decoder.parameters(),
                 'lr': 1e-5},
                {'params': model.tps_stn.parameters(),
                 'lr': 1e-5},
            ], weight_decay=0.05)
            print(f"Resumed at epoch {start_epoch} | "
                  f"Best WER so far: {best_val_wer:.2f}%")

    for epoch in range(start_epoch, epochs + 1):

        # Unfreeze Swin at epoch 4
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(),
                 'lr': 1e-6},
                {'params': model.swin_encoder.proj.parameters(),
                 'lr': lr},
                {'params': model.bilstm.parameters(),
                 'lr': lr},
                {'params': model.decoder.parameters(),
                 'lr': lr},
                {'params': model.tps_stn.parameters(),
                 'lr': lr},
            ], weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5,
                patience=2, min_lr=1e-6, verbose=True
            )
            swin_unfrozen = True
            print("Swin encoder unfrozen.")

        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels,
                         criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(
                    val_loader,
                    desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens = model.generate(
                    images,
                    max_length   = 25,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id
                )

                # Debug first batch each epoch
                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(
                            tokens[i],
                            skip_special_tokens=True
                        )
                        print(f"Target: '{text_labels[i]}' "
                              f"| Pred: '{raw.strip()}'")
                    print("----------------------------\n")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(
                        tokens[i], skip_special_tokens=True
                    )
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        scheduler.step(current_wer)
        print(f"Epoch {epoch} | "
              f"CER: {metrics['CER']:.2f}% | "
              f"WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer':             best_val_wer,
                'cer':                  metrics['CER']
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | WER: {best_val_wer:.2f}%")
        
        # Check early stopping condition
        if early_stopper(current_wer):
            print(f"\nEarly stopping triggered! Training stopped at epoch {epoch}.")
            break
            
        print("=" * 70)


if __name__ == "__main__":
    run_training_pipeline(epochs=80, batch_size=32, lr=5e-5, early_stopping_patience=8, resume_from=CHECKPOINT_PATH)

Loading RoBERTa tokenizer...
BOS=0 | EOS=2 | PAD=1
Registered 38305 samples.
Loading DistilRoBERTa pretrained weights...
DistilRoBERTa: all 6 layers loaded successfully.
Total params    : 191.17M
Trainable params: 191.17M
Swin frozen for first 3 epochs.
Lexicon built: 6231 words.

Training on: cuda
Resuming from: /home/mca24-26/handwritten text detection/iam_htr_distil_2.pth


/tmp/ipykernel_4085019/1126492461.py:619: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(resume_from, map_location=device)


Resumed at epoch 50 | Best WER so far: 21.67%


Epoch 50/80 [Train]: 100%|███████████████████████████████| 1077/1077 [03:10<00:00,  5.66it/s, loss=1.4536]


Epoch 50 | Train Loss: 1.4423


...

Epoch 64 | Train Loss: 1.4364


Epoch 64/80 [Val]:   1%|▍                                                 | 1/120 [00:08<16:45,  8.45s/it]


--- Debug Epoch 64 ---
Target: 'any' | Pred: 'any'
Target: 'any' | Pred: 'any'
Target: 'unopposed' | Pred: 'impressed'
----------------------------



Epoch 64/80 [Val]: 100%|████████████████████████████████████████████████| 120/120 [16:09<00:00,  8.08s/it]

Epoch 64 | CER: 13.43% | WER: 21.74%
EarlyStopping counter: 8 out of 8

Early stopping triggered! Training stopped at epoch 64.


In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForCausalLM
import timm
import Levenshtein
from tqdm import tqdm

# ============================================================================
# 1. Model definitions (exactly matching training)
# ============================================================================

class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4,4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128*4*4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points*2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:,:,0].mean(dim=1)
        cy = control_points[:,:,1].mean(dim=1)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:,0,0] = 1.0
        theta[:,1,1] = 1.0
        theta[:,0,2] = torch.tanh(cx) * 0.05
        theta[:,1,2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=False,
            features_only=True,
            img_size=(64,256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.proj(feat)

class DistilRoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        config = RobertaConfig.from_pretrained("distilroberta-base")
        config.is_decoder = True
        config.add_cross_attention = True
        config.vocab_size = vocab_size
        self.decoder = RobertaForCausalLM(config)

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(input_ids=input_ids, encoder_hidden_states=encoder_hidden_states, labels=labels)

class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model//2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.decoder = DistilRoBERTaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory

    @torch.no_grad()
    def generate(self, images, max_length=25, bos_token_id=0, eos_token_id=2, beam_size=5):
        """
        Batched beam search – processes all images in parallel.
        Returns tensor of shape (B, seq_len) without BOS.
        """
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)  # (B, T, d)
        
        # Expand memory and beams: each original sample becomes beam_size copies
        memory = memory.repeat_interleave(beam_size, dim=0)  # (B*beam_size, T, d)
        
        # Initialize sequences: each beam starts with BOS
        sequences = torch.full((B * beam_size, 1), bos_token_id, dtype=torch.long, device=device)
        log_probs = torch.zeros(B * beam_size, device=device)
        finished = torch.zeros(B * beam_size, dtype=torch.bool, device=device)
        
        for step in range(max_length - 1):
            if finished.all():
                break
            # Forward all beams together
            outputs = self.decoder(
                input_ids=sequences,
                encoder_hidden_states=memory,
                labels=None
            )
            logits = outputs.logits[:, -1, :]  # (B*beam_size, vocab)
            step_log_probs = F.log_softmax(logits, dim=-1)  # (B*beam_size, vocab)
            
            # Add current cumulative log prob
            combined = log_probs.unsqueeze(1) + step_log_probs  # (B*beam_size, vocab)
            # For finished beams, set to -inf so they are never chosen
            combined[finished] = -1e9
            
            # Reshape to (B, beam_size * vocab) to select top beam_size per original sample
            combined = combined.view(B, beam_size * step_log_probs.size(-1))  # (B, beam_size * vocab)
            top_log_probs, flat_indices = combined.topk(beam_size, dim=-1)  # (B, beam_size)
            
            # Decode flat indices to (beam_index, token_id)
            vocab_size = step_log_probs.size(-1)
            prev_beam_idx = flat_indices // vocab_size  # (B, beam_size)
            new_tokens = flat_indices % vocab_size      # (B, beam_size)
            
            # Update log_probs: flatten back to (B*beam_size)
            log_probs = top_log_probs.view(B * beam_size)
            
            # Build new sequences: collect previous sequences using prev_beam_idx
            # Map original batch element and old beam index to flat index
            batch_indices = torch.arange(B, device=device).unsqueeze(1).expand(-1, beam_size)  # (B, beam_size)
            old_flat_indices = (batch_indices * beam_size + prev_beam_idx).view(B * beam_size)  # (B*beam_size)
            sequences = sequences[old_flat_indices]  # (B*beam_size, step+1)
            new_tokens_flat = new_tokens.view(B * beam_size, 1)
            sequences = torch.cat([sequences, new_tokens_flat], dim=1)
            
            # Update finished flags
            finished = finished[old_flat_indices] | (new_tokens_flat.squeeze(1) == eos_token_id)
        
        # After loop, for each original sample choose beam with highest log_prob
        log_probs = log_probs.view(B, beam_size)
        best_beam_idx = log_probs.argmax(dim=-1)  # (B,)
        sequences = sequences.view(B, beam_size, -1)
        best_sequences = sequences[torch.arange(B), best_beam_idx]  # (B, seq_len)
        # Remove BOS token
        return best_sequences[:, 1:]

# ============================================================================
# 2. Paths and setup
# ============================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_distil_2.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
print(f"BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}")

model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} | Best WER: {checkpoint['best_wer']:.2f}%")

# ============================================================================
# 3. Preprocessing
# ============================================================================
val_transform = T.Compose([
    T.Resize((64, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    tensor = val_transform(image).unsqueeze(0)
    return tensor.to(device)

# ============================================================================
# 4. Single image prediction
# ============================================================================
def predict_single_image(image_path, beam_size=5):
    input_tensor = preprocess_image(image_path)
    with torch.no_grad():
        tokens = model.generate(
            input_tensor,
            max_length=25,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            beam_size=beam_size
        )
    return tokenizer.decode(tokens[0], skip_special_tokens=True)

# ============================================================================
# 5. Validation evaluation
# ============================================================================
def get_validation_loader(batch_size=32):
    from torch.utils.data import Dataset, DataLoader
    class SimpleValDataset(Dataset):
        def __init__(self, samples, transform):
            self.samples = samples
            self.transform = transform
        def __len__(self): return len(self.samples)
        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            image = Image.open(img_path).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, text
    full_samples = []
    with open(WORDS_TXT_FILE, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip(): continue
            parts = line.strip().split()
            if len(parts) < 9 or parts[1] != "ok": continue
            transcription = " ".join(parts[8:]).strip()
            if not transcription: continue
            word_id = parts[0]
            id_parts = word_id.split('-')
            img_path = os.path.join(
                BASE_DATA_DIR, "words", id_parts[0],
                f"{id_parts[0]}-{id_parts[1]}", f"{word_id}.png"
            )
            if os.path.exists(img_path):
                full_samples.append((img_path, transcription))
    total = len(full_samples)
    train_size = int(0.9 * total)
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(total, generator=generator).tolist()
    val_indices = indices[train_size:]
    val_samples = [full_samples[i] for i in val_indices]
    val_dataset = SimpleValDataset(val_samples, val_transform)
    return DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

def evaluate_validation(beam_size=5):
    val_loader = get_validation_loader(batch_size=32)
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, texts in tqdm(val_loader, desc="Evaluating"):
            images = images.to(device)
            tokens = model.generate(
                images,
                max_length=25,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                beam_size=beam_size
            )
            for i in range(len(texts)):
                pred = tokenizer.decode(tokens[i], skip_special_tokens=True)
                all_preds.append(pred)
                all_targets.append(texts[i])
    total_cer = 0.0
    wrong_words = 0
    for p, t in zip(all_preds, all_targets):
        d = Levenshtein.distance(p.strip(), t.strip())
        total_cer += d / max(len(t), 1)
        if d != 0:
            wrong_words += 1
    n = len(all_targets)
    cer = (total_cer / n) * 100
    wer = (wrong_words / n) * 100
    print(f"test CER: {cer:.2f}% | WER: {wer:.2f}%")
    return cer, wer

# ============================================================================
# 6. Run evaluation
# ============================================================================
evaluate_validation(beam_size=5)

Using device: cuda
BOS=0, EOS=2, PAD=1


/tmp/ipykernel_236877/1101777984.py:183: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)


Loaded checkpoint from epoch 62 | Best WER: 21.43%


Evaluating: 100%|███████████████████████████████████████████████████████| 120/120 [00:14<00:00,  8.31it/s]

test CER: 13.70% | WER: 22.16%


(13.704102026341635, 22.16131558339859)